# ipynb import and export

> Reading and writing dialogs as Jupyter notebooks

In [ ]:
#| default_exp ipynb

Dialogs are stored as Jupyter notebooks (`.ipynb` files), so any notebook tool can open them. This module handles the conversion:

| Message Type | Cell Type | Notes |
|-------------|-----------|-------|
| `note` | markdown | Direct mapping |
| `prompt` | markdown | Has `solveit_ai` metadata; content + AI response joined with separator |
| `code` | code | Outputs preserved as standard notebook outputs |
| `raw` | raw | Direct mapping |

**Writing** converts `Dialog` -> `.ipynb`, **Reading** does the reverse.

In [ ]:
#| export
from fastcore.utils import *
from fastcore.xtras import atomic_save
import nbformat,json
from nbformat.v4 import new_markdown_cell as new_md, new_code_cell as new_code, new_raw_cell as new_raw, new_notebook
from base64 import b64encode,b64decode
from nbformat.validator import NotebookValidationError
from nbformat.reader import NotJSONError
from llmsurgery.dialog import *

In [ ]:
import random
from tempfile import mkdtemp
from fastcore.test import *

In [ ]:
random.seed(7)
tstdir = Path(mkdtemp())
dlg = Dialog('dlg')
nt_msg = dlg.mk_message('A *test* dialog', msg_type=snote)
nt_msg.mk_attachment(b'not really a png', 'image/png')
code_msg = dlg.mk_message('1+1', msg_type=scode, output=code_output('2'))
ai_msg = dlg.mk_message('Add them.', msg_type=sprompt, output='The answer is **2**.')
raw_msg = dlg.mk_message('plain text', msg_type=sraw)
dlg

<div class="prose" markdown="1">


**dlg**


<details markdown='1'>

- A *test* dialog
- 1+1 ⇒ [{'output_type': 'execute_result', 'metadata': {}, 'data': {'text/plain': '2'}, 'execution_count': 1}]
- Add them. ⇒ [{'output_type': 'display_data', 'metadata': {'is_ai_res': True}, 'data': {'text/markdown': 'The answer is **2**.'}}]
- plain text

</details>

</div>

## Writing

Attachments look like this:
```js
  {
   "attachments": {
    "image.png": {
     "image/png": "iVBO...kJggg=="
    }
  },
```

In [ ]:
#| export
def att2dict(att): return {att.content_type: b64encode(att.data).decode('ascii') if isinstance(att.data, bytes) else att.data}

Prompts are special: they contain both user input *and* AI response in a single markdown cell. We:

1. Mark them with `solveit_ai: true` in cell metadata
2. Join content + response with a separator (`reply_sep`) when writing
3. Split on that separator when reading back

The separator includes a hidden HTML comment to avoid collisions with normal content.

In [ ]:
#| export
reply_sep = "\n\n##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->\n\n"

def split_cell_src(cell):
    "Split cell source into (content, ai_reply_or_None)"
    content, *reply = cell.get('source', '').split(reply_sep)
    return content, (reply[0] if reply else None)

In [ ]:
#| export
_out_meta_skip = {'__type'}

def _clean_out_meta(o):
    if m := o.get('metadata'): o['metadata'] = {k:v for k,v in m.items() if k not in _out_meta_skip}
    return o

In [ ]:
#| export
@patch
def cell_meta(self:Message):
    "Metadata dict to write: `meta` plus demoted `meta_attrs` fields, falsy values omitted"
    meta = dict(self.meta)
    for a,k in self.meta_attrs.items():
        if (v := getattr(self, a, None)): meta[k] = v
        else: meta.pop(k, None)
    return meta

`cell_meta` assembles everything `to_cell` will write as cell metadata, and is the override point when a host's in-memory types differ from the file's: convert after `super()`, before nbformat validates the cell (solveit stores its UI flags as ints, but the ipynb schema types `collapsed`/`hide_input` as boolean).

In [ ]:
#| export
@patch
def to_cell(self:Message, version=2):
    "Convert message to an nbformat cell"
    meta = self.cell_meta()
    cts = self.content or ''
    f = new_md if self.msg_type in (snote,sprompt) else new_code if self.msg_type == scode else new_raw
    if self.msg_type==sprompt:
        meta['solveit_ai'] = True
        if o := self.ai_res: cts = cts + reply_sep + o
    outkw = {}
    if self.msg_type==scode and self.output:
        outputs = self.output
        if version==1: outputs = json.loads(outputs)
        outkw['outputs'] = [nbformat.from_dict(_clean_out_meta(o)) for o in outputs]
    atts = {att.id: att2dict(att) for att in (self.attachments or [])}
    if atts and self.msg_type in (sprompt,snote): outkw['attachments'] = atts
    id_ = self.id[1:]
    try: return f(cts, id=id_, metadata=meta, **outkw)
    except NotebookValidationError as e:
        print(f"NB validation err: {e}")
        outkw['outputs'] = [] if self.msg_type == scode else ''
        return f(cts, id=id_, metadata=meta, **outkw)

In [ ]:
test_eq(Message().to_cell()['metadata'],{})

`meta_attrs` is how a host teaches serialization about its own fields without this library knowing them: declare attribute → metadata key, and `cell_meta`/`cell2msg` demote/promote them, with falsy values omitted from the file. Anything *not* declared still round-trips untouched inside `meta`:

In [ ]:
class NoteMsg(Message): meta_attrs = dict(bookmark='bookmark')
class NoteDlg(Dialog): msg_cls = NoteMsg

bookmark_cell = NoteMsg(bookmark=9).to_cell()
test_eq(bookmark_cell['metadata']['bookmark'], 9)
test_eq(NoteMsg().to_cell()['metadata'], {})  # default/absent values aren't written

When a host's in-memory type differs from what the file should carry (the ipynb schema types some keys, and validation runs as the cell is built), it converts in a `cell_meta` override:

In [ ]:
class FlagMsg(Message):
    meta_attrs = dict(collapsed='collapsed')
    def cell_meta(self): return {k: bool(v) for k,v in super().cell_meta().items()}

assert FlagMsg(collapsed=1).to_cell()['metadata']['collapsed'] is True

In [ ]:
# Prompts serialize as markdown with the reply appended after `reply_sep`
pr_msg = dlg.mk_message('What is 2+2?', output='The answer is 4.', msg_type='prompt')
pr_cell = pr_msg.to_cell()
test_eq(pr_cell['cell_type'], 'markdown')
test_eq(pr_cell['metadata']['solveit_ai'], True)
assert 'What is 2+2?' in pr_cell['source']
assert reply_sep in pr_cell['source']
assert 'The answer is 4.' in pr_cell['source']

# A prompt without a reply gets no separator
pr_empty = dlg.mk_message('Hello?', output='', msg_type='prompt')
assert reply_sep not in pr_empty.to_cell()['source']


In [ ]:
#| export
def get_ipynb(dlg:Dialog, version=2, msgs=None):
    "Notebook object for `dlg`; `msgs` defaults to all its messages"
    cells = [m.to_cell(version=version) for m in (dlg.messages if msgs is None else msgs)]
    return new_notebook(cells=cells, metadata=dict(dlg.meta))


In [ ]:
#| export
def write_ipynb(dlg:Dialog, fname=None, version=2, msgs=None, **kwargs):
    "Write `dlg` as a notebook, or return the JSON string if `fname` is None; `kwargs` (e.g. `uid`/`gid`) pass to `atomic_save`"
    nb = get_ipynb(dlg, version=version, msgs=msgs)
    if not fname: return nbformat.writes(nb, indent=2)
    with atomic_save(Path(fname), mode='w', encoding='utf-8', **kwargs) as f: nbformat.write(nb, f)

In [ ]:
write_ipynb(dlg, fname=tstdir/'dlg.ipynb', version=2)

In [ ]:
#| export
@patch
def write(self:Dialog, base_path, version=2, msgs=None, **kwargs):
    write_ipynb(self, Path(base_path)/f'{self.name}.ipynb', version=version, msgs=msgs, **kwargs)

In [ ]:
dlg.write(tstdir)

In [ ]:
#| export
def ipynb_cells(path, nm, prefix=None, suffix=None):
    tmpl = path/f'{nm}.ipynb'
    if not tmpl.exists(): return []
    try: cells = nbformat.read(tmpl, as_version=nbformat.NO_CONVERT).cells
    except NotJSONError: return []
    return listify(prefix) + cells + listify(suffix)

In [ ]:
test_eq(len(ipynb_cells(tstdir, dlg.name)), len(dlg.messages))
ipynb_cells(tstdir, dlg.name)[0]

{'attachments': {'2530bb1d-6d13-4cde-8623-7b2ed91e3f72': {'image/png': 'bm90IHJlYWxseSBhIHBuZw=='}},
 'cell_type': 'markdown',
 'id': 'a54dca18',
 'metadata': {},
 'source': 'A *test* dialog'}

## Reading

In [ ]:
#| export
def dict2att(att_id, att_data):
    "Convert attachment dict to Attachment object"
    content_type, data = first(att_data.items())
    if isinstance(data, str): data = b64decode(data)
    return Attachment(data, content_type, att_id)

def _output_from_cell(cell):
    if cell.cell_type!='code': return ''
    return [nbformat.from_dict(o) for o in getattr(cell, 'outputs', [])]

In [ ]:
#| export
@patch
def cell2msg(self:Dialog, cell):
    "Convert single notebook cell to message object"
    meta = dict(cell.metadata)
    kwargs = {a: meta.pop(k) for a,k in self.msg_cls.meta_attrs.items() if k in meta}
    content, reply = split_cell_src(cell)
    msg_type = sprompt if meta.pop('solveit_ai', 0) else snote if cell.cell_type=='markdown' else cell.cell_type
    output = ('' if msg_type in (snote,sraw)  # chkstyle: ignore
        else _output_from_cell(cell) if msg_type == scode
        else prompt_output(reply) if reply else [])
    atts = [dict2att(att_id, att_data) for att_id, att_data in cell.get('attachments', {}).items()]
    id = '_' + (getattr(cell, 'id', None) or rtoken_hex(4))
    return self.msg_cls(content, id=id, output=output, msg_type=msg_type, dlg=self, attachments=atts, meta=meta, **kwargs)

In [ ]:
back = NoteDlg('t').cell2msg(bookmark_cell)
test_eq(back.bookmark, 9)
test_eq(back.meta, {})  # promoted out of `meta` into the attribute

In [ ]:
# Test roundtrip prompts
pr_back = dlg.cell2msg(pr_cell)
test_eq(pr_back.content, 'What is 2+2?')
test_eq(pr_back.msg_type, 'prompt')
test_eq(pr_back.ai_res, 'The answer is 4.')

In [ ]:
#| export
@patch
def from_cells(self:Dialog, cells):
    self.messages = L(cells).map(self.cell2msg)
    return self

def read_ipynb(fname, cls=Dialog, name=None):
    "Read a dialog from notebook file `fname` (`.ipynb` added if missing), constructing via `cls`; `name` defaults to the file stem"
    f = Path(fname)
    if f.suffix != '.ipynb': f = f.with_suffix('.ipynb')
    if not f.exists(): return print(f,'does not exist')
    try: nb = nbformat.read(f, as_version=nbformat.NO_CONVERT)
    except (NotJSONError, nbformat.ValidationError, PermissionError): return
    return cls(name or f.stem, meta=dict(nb.get('metadata', {}))).from_cells(nb.cells)

In [ ]:
dlg = read_ipynb(tstdir/'dlg')
dlg

<div class="prose" markdown="1">


**dlg**


<details markdown='1'>

- A *test* dialog
- 1+1 ⇒ [{'data': {'text/plain': '2'}, 'execution_count': 1, 'metadata': {}, 'output_type': 'execute_result'}]
- Add them. ⇒ [{'output_type': 'display_data', 'metadata': {'is_ai_res': True}, 'data': {'text/markdown': 'The answer is **2**.'}}]
- plain text
- What is 2+2? ⇒ [{'output_type': 'display_data', 'metadata': {'is_ai_res': True}, 'data': {'text/markdown': 'The answer is 4.'}}]
- Hello?

</details>

</div>

### Metadata preservation

The library itself names no host fields: everything in cell metadata that a `meta_attrs` declaration doesn't claim rides verbatim in `Message.meta`, and notebook-level metadata rides in `Dialog.meta` the same way. So a file written by any host survives a read/write round trip through the plain classes with every annotation intact:

In [ ]:
m0 = dlg.messages[0]
m0.meta['my_app_flag'] = dict(level=3)
dlg.meta['my_app'] = dict(version=1)
dlg.write(tstdir)
dlg = read_ipynb(tstdir/'dlg')
test_eq(dlg.messages[0].meta['my_app_flag'], dict(level=3))
test_eq(dlg.meta['my_app'], dict(version=1))

## export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()